# README: silver_facilities (team master table)

**What it is.** One row per healthcare facility in India, cleaned from the Virtue Foundation dataset (DAIS 2026). **9,997 rows x 50 columns.** Every field in the source was scraped from web text, so we treat it as a *claim*: suspicious records are **kept and flagged**, not deleted, and the Gold step turns flags into lower trust.

**Table to use:** workspace.default.silver_facilities  (for Gold, the app, and all analysis)
Older per-person versions (silver_facilities_joyce, *_daniel) are for reference only.

## Pipeline
| Layer | Table(s) | Notebook |
|---|---|---|
| Bronze (raw, unchanged) | bronze_facilities, bronze_india_post_pincode_directory, bronze_nfhs_5_district_health_indicators | 01_bronze_explore, Bronze 01 - Source Inventory, 01_bronze_daniel |
| **Silver master (this)** | **silver_facilities** | **03_silver_master** |
| Gold (next) | trust score per facility x service | 04_gold (to build) |

## Quick start
    df = spark.table("workspace.default.silver_facilities")
    display(df.filter("n_flags = 0"))                      # only records with no quality flags
    display(df.filter("array_contains(specialties, 'nephrology')"))

## Column groups (50)
- **Basic (5):** unique_id, name, facility_type, operator_type, description
- **Service evidence (5):** specialties, procedure, equipment, capability (de-duplicated lists), n_specialties
- **Scale (3):** n_doctors, n_beds, year_established (mostly empty)
- **Address (8):** address, city, state (corrected), state_raw, pincode, pin_district, pin_state, pin_n_districts
- **Location (8):** lat, lon, pin_lat, pin_lon, dist_to_pin_km, **loc_lat, loc_lon** (best location), **loc_source**
- **Contact / sources (6):** official_phone, official_website, email, websites, n_websites, source_urls
- **Flags (9):** flag_bad_coords, flag_state_mismatch, flag_coords_far_from_pin, flag_location_uncertain, flag_duplicate, flag_mixed_identity, flag_too_many_specialties, flag_implausible_counts, n_flags
- **Lineage (6):** source_table, source_version, pin_source_table, pin_source_version, silver_built_at, silver_notebook

Every column also has a description in Catalog (Catalog > workspace > default > silver_facilities > Overview).

## Rules for using this table
1. **Distance:** always use loc_lat / loc_lon, never lat / lon. If flag_location_uncertain is True, show "location approximate".
2. **Evidence:** quote the exact text from description / procedure / equipment / capability / specialties when recommending a facility.
3. **Flags lower trust, they do not delete.** Serious: mixed_identity, state_mismatch, duplicate. Milder: coords_far_from_pin, too_many_specialties.
4. **Do not edit this table by hand.** Change the notebook and press Run all (about 1 minute); integrity checks stop the run if anything is off.

## Key numbers
10,088 raw rows -> 11 exact duplicate rows removed -> 10,077 -> 80 empty records dropped -> **9,997**.
0 flags: 7,854 | 1 flag: 1,639 | 2+ flags: 584.

## Known limitations
- mixed_identity has false positives at medical colleges with an AYUSH department.
- Andhra Pradesh vs Telangana state conflicts may be old addresses, not errors.
- 455 facilities have an uncertain location (coordinates and PIN disagree and the city could not settle it).

## Change log
- v1 (Sat 1:05 PM) 02_silver_joyce: cleaning + flags.
- v1.1 (Sat 1:45 PM) location check vs PIN, city referee, loc_lat / loc_lon.
- **Master (Sat 2:30 PM) 03_silver_master:** + Daniel's pinned Bronze version 0, lineage columns, integrity checks; found and removed 11 exact duplicate rows.

Team log and full data dictionary: Google Doc "DSC 2026".

# 03 Silver MASTER: the team's final cleaned facility table
**Output:** workspace.default.silver_facilities  (use this table for Gold / app / analysis)
**Inputs (pinned):** workspace.default.bronze_facilities VERSION AS OF 0 and workspace.default.bronze_india_post_pincode_directory VERSION AS OF 0

This combines the two team versions:
- **From Joyce's Silver (02_silver_joyce):** all cleaning logic: list parsing + de-duplication, type / number / coordinate fixes, PIN-directory district join with state correction, city-referee location choice, and 8 quality flags.
- **From Daniel's Bronze work (Bronze 01 + 01_bronze_daniel):** reproducibility and provenance: inputs are pinned to the Delta version Daniel verified row-by-row (version 0, 10,088 / 165,627 rows, 0 missing, 0 extra); every row carries lineage columns (source table, source version, build time); integrity checks stop the run if row counts, unique IDs or joins look wrong.

Steps: 1 load pinned Bronze + parse lists, 2 fix basic fields, 3 PIN join + state fix, 4 flags + best location, 5 lineage + integrity checks + save. Flagged records are kept (Gold lowers their trust); only empty records are dropped.

In [0]:
# Step 1: load the PINNED raw data and turn the JSON-text list fields into real Python lists (deduplicated)
import json, re, datetime
import numpy as np
import pandas as pd

BRONZE_FAC = "workspace.default.bronze_facilities"
BRONZE_PIN = "workspace.default.bronze_india_post_pincode_directory"
BRONZE_VERSION = 0   # Delta version Daniel verified row-by-row in 01_bronze_daniel (0 missing, 0 extra)

raw = spark.sql(f"SELECT * FROM {BRONZE_FAC} VERSION AS OF {BRONZE_VERSION}").toPandas()
assert len(raw) == 10088, f"unexpected Bronze row count: {len(raw)}"
print(f"raw rows: {len(raw):,} (from {BRONZE_FAC} VERSION AS OF {BRONZE_VERSION})")

# --- unique_id check (found by the master integrity checks: unique_id is NOT unique in Bronze) ---
dup_id = raw["unique_id"].duplicated(keep=False)
exact_copy = raw.astype(str).duplicated(keep="first")       # identical in all 51 columns
print(f"rows sharing a unique_id: {int(dup_id.sum())} | distinct ids involved: {raw.loc[dup_id, 'unique_id'].nunique()}")
print(f"  exact copies (all 51 columns identical) -> dropped: {int(exact_copy.sum())}")
raw = raw.loc[~exact_copy].reset_index(drop=True)
still_dup = raw["unique_id"].duplicated(keep=False)
print(f"  same id but DIFFERENT content -> kept, flagged: {int(still_dup.sum())}")
if still_dup.any():
    display(raw.loc[still_dup, ["unique_id", "name", "address_city", "address_zipOrPostcode"]]
            .sort_values("unique_id").head(10))
df = raw.copy()

# record_id = truly unique key per row (unique_id, plus -1, -2 ... for the 2nd/3rd record sharing an id)
k = df.groupby("unique_id").cumcount()
df["record_id"] = df["unique_id"].astype(str) + np.where(k > 0, "-" + k.astype(str), "")
df["flag_duplicate_id"] = df["unique_id"].duplicated(keep=False)
assert df["record_id"].is_unique

NULLS = {"", "null", "none", "nan", "[]", "{}"}

def clean_str(v):
    """Return a stripped string, or None for empty / 'null' values."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = str(v).strip()
    return None if s.lower() in NULLS else s

def parse_list(v):
    """'["a","a","b"]' -> ['a', 'b']  (keeps order, drops blanks and case-insensitive duplicates)"""
    s = clean_str(v)
    if s is None:
        return []
    try:
        items = json.loads(s)
        if not isinstance(items, list):
            items = [items]
    except Exception:
        items = [s]
    out, seen = [], set()
    for it in items:
        t = clean_str(it)
        if t and t.lower() not in seen:
            seen.add(t.lower())
            out.append(t)
    return out

def raw_len(v):
    try:
        x = json.loads(v)
        return len(x) if isinstance(x, list) else 1
    except Exception:
        return 0

LIST_COLS = ["specialties", "procedure", "equipment", "capability", "websites",
             "phone_numbers", "source_urls", "affiliationTypeIds"]
before = df["specialties"].apply(raw_len).sum()
for c in LIST_COLS:
    df[c] = df[c].apply(parse_list)
after = df["specialties"].str.len().sum()
print(f"specialty entries: {before:,} before dedup -> {after:,} after dedup")

raw rows: 10,088 (from workspace.default.bronze_facilities VERSION AS OF 0)
rows sharing a unique_id: 22 | distinct ids involved: 11
  exact copies (all 51 columns identical) -> dropped: 11
  same id but DIFFERENT content -> kept, flagged: 0
specialty entries: 278,614 before dedup -> 118,058 after dedup


In [0]:
# Step 2: fix basic fields: text, categories, numbers, coordinates, PIN code
for c in ["name", "description", "address_line1", "address_city", "address_stateOrRegion",
          "address_zipOrPostcode", "facilityTypeId", "operatorTypeId",
          "officialPhone", "officialWebsite", "email"]:
    df[c] = df[c].apply(clean_str)

# facility / operator type: keep only valid categories, fix typos (numbers and URLs become missing)
df["facility_type"] = df["facilityTypeId"].str.lower().replace({"farmacy": "pharmacy"})
df.loc[~df["facility_type"].isin(["hospital", "clinic", "dentist", "doctor", "pharmacy"]), "facility_type"] = None
df["operator_type"] = df["operatorTypeId"].str.lower().replace({"government": "public"})
df.loc[~df["operator_type"].isin(["private", "public"]), "operator_type"] = None

# numbers stored as text -> numeric; impossible values -> missing
df["n_doctors"] = pd.to_numeric(df["numberDoctors"], errors="coerce")
df["n_beds"] = pd.to_numeric(df["capacity"], errors="coerce")
df["year_established"] = pd.to_numeric(df["yearEstablished"], errors="coerce")
df.loc[~df["year_established"].between(1800, 2026), "year_established"] = np.nan
df.loc[~(df["n_doctors"] > 0), "n_doctors"] = np.nan
df.loc[~(df["n_beds"] > 0), "n_beds"] = np.nan

# coordinates must fall inside India's bounding box
df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
df["flag_bad_coords"] = ~(df["lat"].between(6, 37.5) & df["lon"].between(68, 97.5))
df.loc[df["flag_bad_coords"], ["lat", "lon"]] = np.nan

# city / state: consistent casing; PIN code: exactly 6 digits
df["city"] = df["address_city"].str.strip().str.title()
df["state"] = df["address_stateOrRegion"].str.strip().str.title()
df["pincode"] = df["address_zipOrPostcode"].astype(str).str.extract(r"(\d{6})")[0]

print("valid facility_type:", df["facility_type"].notna().sum(), "| valid operator_type:", df["operator_type"].notna().sum())
print("bad / missing coordinates:", int(df["flag_bad_coords"].sum()), "| valid PIN codes:", df["pincode"].notna().sum())
print("doctors known:", df["n_doctors"].notna().sum(), "| beds known:", df["n_beds"].notna().sum(), "| year known:", df["year_established"].notna().sum())

valid facility_type: 9931 | valid operator_type: 9302
bad / missing coordinates: 124 | valid PIN codes: 9781
doctors known: 3615 | beds known: 2498 | year known: 4768


In [0]:
# Step 3: add district + official state from the India Post PIN directory (pinned version)
# The raw file has one row per POST OFFICE, so one PIN can appear many times.
# We aggregate to ONE row per PIN in Spark first, otherwise the join would duplicate facilities.
pin_agg = spark.sql(f"""
WITH p AS (
  SELECT regexp_extract(CAST(pincode AS STRING), '([0-9]{{6}})', 1) AS pincode,
         initcap(district) AS district, initcap(statename) AS state,
         try_cast(latitude AS DOUBLE) AS lat, try_cast(longitude AS DOUBLE) AS lon
  FROM {BRONZE_PIN} VERSION AS OF {BRONZE_VERSION}),
counts AS (   -- most common district/state for each PIN
  SELECT pincode, district, state, COUNT(*) AS n,
         ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY COUNT(*) DESC) AS rk
  FROM p WHERE pincode <> '' GROUP BY pincode, district, state),
stats AS (
  SELECT pincode, COUNT(DISTINCT district) AS pin_n_districts,
         percentile_approx(lat, 0.5) AS pin_lat, percentile_approx(lon, 0.5) AS pin_lon
  FROM p WHERE pincode <> '' GROUP BY pincode)
SELECT c.pincode, c.district AS pin_district, c.state AS pin_state,
       s.pin_n_districts, s.pin_lat, s.pin_lon
FROM counts c JOIN stats s USING (pincode)
WHERE c.rk = 1
""").toPandas()
assert pin_agg["pincode"].is_unique, "PIN lookup is not one row per PIN"

df = df.drop(columns=[c for c in pin_agg.columns if c != "pincode" and c in df.columns])
df = df.merge(pin_agg, on="pincode", how="left")
assert len(df) == len(raw), "join changed the row count!"

# --- Fix the state field ---
# Many records put a CITY or DISTRICT in the state field (e.g. "Mumbai", "Pune", "Kollam"),
# or use an alias ("Nct Of Delhi"). Those are fixed, not flagged.
norm = lambda s: re.sub(r"[^a-z]", "", s.lower()) if isinstance(s, str) else None
VALID_STATES = {norm(s) for s in pin_agg["pin_state"].dropna().unique()}
ALIASES = {"nctofdelhi": "Delhi", "nctdelhi": "Delhi", "newdelhi": "Delhi", "delhincr": "Delhi",
           "orissa": "Odisha", "pondicherry": "Puducherry", "uttaranchal": "Uttarakhand",
           "jammukashmir": "Jammu And Kashmir"}

def fix_state(row):
    k = norm(row["state"])
    if k in ALIASES:
        return ALIASES[k]
    if k in VALID_STATES:
        return row["state"]
    # not a real state name -> use the PIN code's state (or missing if PIN unknown)
    return row["pin_state"] if isinstance(row["pin_state"], str) else None

df["state_raw"] = df["state"]
df["state"] = df.apply(fix_state, axis=1)
fixed = (df["state"].map(norm) != df["state_raw"].map(norm)) & df["state_raw"].notna()

# A REAL conflict: the record names a valid state, but its PIN code belongs to a different state
df["flag_state_mismatch"] = (df["pin_state"].notna() & df["state"].notna()
                             & (df["state"].map(norm) != df["pin_state"].map(norm)))

print("unique PINs in directory:", len(pin_agg))
print("facilities matched to a district:", df["pin_district"].notna().sum(), "of", len(df))
print("PINs that span more than one district:", int((df["pin_n_districts"] > 1).sum()))
print("state field fixed (held a city/district or an alias):", int(fixed.sum()))
print("real state conflicts (flagged):", int(df["flag_state_mismatch"].sum()))

unique PINs in directory: 19586
facilities matched to a district: 9563 of 10077
PINs that span more than one district: 513
state field fixed (held a city/district or an alias): 382
real state conflicts (flagged): 67


In [0]:
# Step 4: flag suspicious records (we KEEP them, but Gold will trust them less)
text = (df["description"].fillna("") + " | " + df["capability"].apply(" | ".join)
        + " | " + df["procedure"].apply(" | ".join)).str.lower()
df["n_specialties"] = df["specialties"].str.len()
df["n_websites"] = df["websites"].str.len()   # kept as a number for Gold (long lists = noisier evidence), not a flag

# (a) empty record: no name, or no description / specialties / capability at all -> the only rows we drop
df["flag_empty"] = df["name"].isna() | (df["description"].isna() & (df["n_specialties"] == 0)
                                        & (df["capability"].str.len() == 0))

# (b) duplicate: same normalized name in the same city
df["_name_key"] = df["name"].fillna("").str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
df["flag_duplicate"] = (df["_name_key"] != "") & df.duplicated(subset=["_name_key", "city"], keep=False)

# (c) mixed identity: text describes a small single-system practice AND a large hospital
#     (sign that two different places were merged into one record, e.g. Dr Hedgewar Hospital)
small_practice = text.str.contains(r"homeopath|ayurved|unani|siddha|veterinar|dental clinic", regex=True)
large_hospital = (df["n_beds"] >= 100) | text.str.contains(r"open heart|cardiac surgery|neurosurgery|cath lab", regex=True)
df["flag_mixed_identity"] = small_practice & large_hospital

# (d) specialty inflation: 25+ distinct specialties but fewer than 50 beds known
df["flag_too_many_specialties"] = (df["n_specialties"] >= 25) & ~(df["n_beds"] >= 50)

# (e) implausible counts: huge numbers, or far more doctors than beds
df["flag_implausible_counts"] = ((df["n_doctors"] > 3000) | (df["n_beds"] > 5000)
                                 | ((df["n_beds"] >= 10) & (df["n_doctors"] > 5 * df["n_beds"])))

# (f) location check: facility coordinates vs. centre of its PIN code area (haversine distance, km)
def km(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = (np.radians(x) for x in (la1, lo1, la2, lo2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

df["dist_to_pin_km"] = km(df["lat"], df["lon"], df["pin_lat"], df["pin_lon"]).round(1)
df["flag_coords_far_from_pin"] = df["dist_to_pin_km"] > 50

# When they disagree, a third source is the referee: the median location of facilities in the same city.
# (Diagnostic showed both sides are wrong sometimes: ~690 times the PIN, ~526 times the coordinates.)
city_centre = (df[df["lat"].notna()].groupby("city")[["lat", "lon"]].median()
               .rename(columns={"lat": "city_lat", "lon": "city_lon"}))
df = df.drop(columns=["city_lat", "city_lon"], errors="ignore").join(city_centre, on="city")
own_near_city = km(df["lat"], df["lon"], df["city_lat"], df["city_lon"]) <= 25
pin_near_city = km(df["pin_lat"], df["pin_lon"], df["city_lat"], df["city_lon"]) <= 25
has_own, has_pin, far = df["lat"].notna(), df["pin_lat"].notna(), df["flag_coords_far_from_pin"]

conditions = [
    has_own & has_pin & ~far,                    # both agree
    has_own & far & own_near_city & ~pin_near_city,   # PIN looks wrong
    has_pin & far & pin_near_city & ~own_near_city,   # coordinates look wrong
    has_own & ~has_pin,                          # nothing to check against
    ~has_own & has_pin,                          # only the PIN is available
    has_own & far,                               # conflict nobody can settle
]
sources = ["coordinates_match_pin", "coordinates_match_city", "pin_matches_city",
           "coordinates_unchecked", "pin_only", "uncertain"]
df["loc_source"] = np.select(conditions, sources, default="unknown")
use_pin = df["loc_source"].isin(["pin_matches_city", "pin_only"])
df["loc_lat"] = np.where(use_pin, df["pin_lat"], df["lat"])
df["loc_lon"] = np.where(use_pin, df["pin_lon"], df["lon"])
df["flag_location_uncertain"] = df["loc_source"].isin(["uncertain", "unknown"])

flag_cols = ["flag_bad_coords", "flag_state_mismatch", "flag_coords_far_from_pin", "flag_location_uncertain",
             "flag_duplicate", "flag_mixed_identity", "flag_too_many_specialties", "flag_implausible_counts"]
df["n_flags"] = df[flag_cols].sum(axis=1)

print("empty records (will be dropped):", int(df["flag_empty"].sum()))
for c in flag_cols:
    print(f"{c:28s} {int(df[c].sum()):5d}")
print("records with 0 flags:", int((df["n_flags"] == 0).sum()), "| 1 flag:", int((df["n_flags"] == 1).sum()),
      "| 2+ flags:", int((df["n_flags"] >= 2).sum()))
d = df["dist_to_pin_km"].dropna()
print(f"coords vs PIN centre: {len(d):,} comparable | median {d.median():.1f} km | within 10 km {(d <= 10).mean():.0%} | over 50 km {(d > 50).sum()}")
print("location source:", df["loc_source"].value_counts().to_dict())

empty records (will be dropped): 80
flag_bad_coords                124
flag_state_mismatch             67
flag_coords_far_from_pin      1581
flag_location_uncertain        455
flag_duplicate                  16
flag_mixed_identity            156
flag_too_many_specialties      436
flag_implausible_counts          7
records with 0 flags: 7854 | 1 flag: 1639 | 2+ flags: 584
coords vs PIN centre: 9,528 comparable | median 2.4 km | within 10 km 73% | over 50 km 1581
location source: {'coordinates_match_pin': 7947, 'coordinates_match_city': 689, 'pin_matches_city': 526, 'coordinates_unchecked': 425, 'uncertain': 366, 'unknown': 89, 'pin_only': 35}


In [0]:
# Step 5: lineage columns (Daniel) + integrity checks + save the MASTER Silver table
OUTPUT = "workspace.default.silver_facilities"
rename = {"address_line1": "address", "officialPhone": "official_phone",
          "officialWebsite": "official_website"}
keep = ["unique_id", "name", "facility_type", "operator_type", "description",
        "specialties", "procedure", "equipment", "capability",
        "n_specialties", "n_doctors", "n_beds", "year_established",
        "address_line1", "city", "state", "state_raw", "pincode", "pin_district", "pin_state", "pin_n_districts",
        "lat", "lon", "pin_lat", "pin_lon", "dist_to_pin_km", "loc_lat", "loc_lon", "loc_source",
        "officialPhone", "officialWebsite", "email", "websites", "n_websites", "source_urls",
        ] + flag_cols + ["n_flags"]

silver = df.loc[~df["flag_empty"], keep].rename(columns=rename).reset_index(drop=True)

# --- Provenance / lineage: where every row came from and when it was built ---
silver["source_table"] = BRONZE_FAC
silver["source_version"] = BRONZE_VERSION
silver["pin_source_table"] = BRONZE_PIN
silver["pin_source_version"] = BRONZE_VERSION
silver["silver_built_at"] = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
silver["silver_notebook"] = "DSC Hack 2026 / 03_silver_master"

# --- Integrity checks: stop before saving if anything looks wrong ---
n_empty = int(df["flag_empty"].sum())
assert len(silver) == len(raw) - n_empty, "row count != raw - empty records"
assert silver["unique_id"].is_unique, "duplicate unique_id after cleaning"
assert silver["unique_id"].notna().all(), "missing unique_id"
assert set(silver["unique_id"]) <= set(raw["unique_id"]), "unique_id not found in Bronze"
assert silver[flag_cols].isin([True, False]).all().all(), "flag columns must be True/False"
assert silver["loc_lat"].dropna().between(6, 37.5).all() and silver["loc_lon"].dropna().between(68, 97.5).all(), \
       "best location outside India"
print("integrity checks passed")

(spark.createDataFrame(silver)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(OUTPUT))
spark.sql(f"""COMMENT ON TABLE {OUTPUT} IS 'MASTER cleaned facilities (team version). Built by 03_silver_master from {BRONZE_FAC} and {BRONZE_PIN} VERSION AS OF {BRONZE_VERSION}. Flagged records kept; see flag_* and n_flags.'""")

check = spark.table(OUTPUT)
version = spark.sql(f"DESCRIBE HISTORY {OUTPUT} LIMIT 1").first()["version"]
print(f"saved {OUTPUT}: {check.count():,} rows x {len(check.columns)} columns (Delta version {version})")
print(f"rows: {len(raw):,} raw -> {len(silver):,} silver (dropped {n_empty} empty records)")
display(check.limit(5))

/home/spark-c541c668-71be-45b5-959b-76/.ipykernel/75/command-7846826793504422-1770310677:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  silver["silver_built_at"] = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


integrity checks passed
saved workspace.default.silver_facilities: 9,997 rows x 50 columns (Delta version 1)
rows: 10,077 raw -> 9,997 silver (dropped 80 empty records)


unique_id,name,facility_type,operator_type,description,specialties,procedure,equipment,capability,n_specialties,n_doctors,n_beds,year_established,address,city,state,state_raw,pincode,pin_district,pin_state,pin_n_districts,lat,lon,pin_lat,pin_lon,dist_to_pin_km,loc_lat,loc_lon,loc_source,official_phone,official_website,email,websites,n_websites,source_urls,flag_bad_coords,flag_state_mismatch,flag_coords_far_from_pin,flag_location_uncertain,flag_duplicate,flag_mixed_identity,flag_too_many_specialties,flag_implausible_counts,n_flags,source_table,source_version,pin_source_table,pin_source_version,silver_built_at,silver_notebook
268873b8-f9d5-49d2-bc63-b29ed7512504,Kovilpatti Venkateswara Hospitals,hospital,private,"The Complete Orthopedic Care Hospital In Kovilpatti. Leading center for Joint Replacement Surgery, Complex trauma, Deformity correction and Spine surgery; Pioneer in pinhole surgery and keyhole surgery; Charitable Foundation providing free surgeries.","List(orthopedicSurgery, spineNeurosurgery, jointReplacementSurgery, Oromaxillo Facial and Reconstructive Surgery, plasticSurgery, reconstructivePlasticSurgery, traumaAndFractureSurgery, physicalMedicineAndRehabilitation, painMedicinePMR, internalMedicine, kneeReplacementSurgery, oralAndMaxillofacialSurgery, emergencyMedicine, criticalCareMedicine)","List(Spinal Disc Problems treatment, Cervical Spinal Stenosis treatment, Lumbar Spinal Stenosis treatment, Spinal Cord Trauma treatment, Congenital Disorders of the Spinal Cord treatment, Spine Infection treatment, Vascular Malformations of Spine treatment, Knee Arthritis treatment, Knee Arthritis Care, Knee Stem Cell Treatment, Meniscus Tears treatment, Cartilage Defects treatment, Anterior Cruciate Ligament Tears treatment, Posterior Cruciate Ligament Tear treatment, Medial Patello-Femoral Ligament Tears treatment, Back Pain treatment, Spondylosis/Spondylitis treatment, Slip/Slipped Disc treatment, Pain after Surgery treatment, Nerve Pain treatment, Head Ache treatment, Cancer Pain treatment, Arthritis treatment, Knee Pain treatment, Neck Pain treatment, Spinal Surgeries, Endospinal Surgeries, Joint Replacement Surgeries, Oromaxillo Facial and Reconstructive Surgery, Plastic and Reconstructive Surgeries, Trauma and Fracture Surgery, Rehabilitation, Knee Replacement, Arthroscopy, Pain Management, Provides fracture care, Offers fracture treatment, Provides orthopedic emergency care, Operates a Trauma & Emergency Unit that is ready 24/7, Provides comprehensive orthopedic care under one roof)",List(),"List(Leading center for Joint Replacement Surgery, Complex trauma, Deformity correction and Spine surgery, Pioneer in Pinhole Surgery and Keyhole Surgery, World Class Specialties in Spine/ Endospine Surgeries, Joint Replacement Surgeries, Orthopedics and Trauma Care, Equipped with the most modern and infection-free recuperating facility, Charitable Foundation offering free surgeries, Rehabilitation and Physiotherapy services, Established in 1996, 24/7 trauma and emergency care, Trauma & Emergency Unit, Comprehensive total orthopedic care under one roof, Offers advanced diagnostic and treatment options for major orthopedic conditions)",14,null,null,null,"520, Tirunelveli Nh",Kovilpatti,Tamil Nadu,Tamil Nadu,628503,Tuticorin,Tamil Nadu,1.0,9.165903091430664,77.85116577148438,9.1486,77.7942,6.5,9.165903091430664,77.85116577148438,coordinates_match_pin,+919442648552,kovilpattivenkateswarahospitals.com,kvpvenkateswarahosp@hotmail.com,"List(https://www.kovilpattivenkateswarahospitals.com/, http://kovilpattivenkateswarahospitals.com/, https://www.cybo.com/IN-biz/dr-sennakesvan-hospital, kovilpattivenkateswarahospitals.com, https://www.facebook.com/kovilpattivenkateswarahospital/, https://in.linkedin.com/company/kovilpatti-venkateswara-hospitals, http://kovilpattivenkateswarahospitals.com)",7,"List(https://www.kovilpattivenkateswarahospitals.com/, http://kovilpattivenkateswarahospitals.com/, https://www.cybo.com/IN-biz/dr-sennakesvan-hospi

In [0]:
# Diagnostic: when a facility's coordinates and its PIN code disagree (> 50 km), which one is wrong?
# Referee = city_lat / city_lon from Step 4: the median location of facilities in the same city.
far = df[df["flag_coords_far_from_pin"]].copy()
far["own_to_city_km"] = km(far["lat"], far["lon"], far["city_lat"], far["city_lon"]).round(1)
far["pin_to_city_km"] = km(far["pin_lat"], far["pin_lon"], far["city_lat"], far["city_lon"]).round(1)

own_ok = far["own_to_city_km"] <= 25
pin_ok = far["pin_to_city_km"] <= 25
print("records where coordinates and PIN disagree:", len(far))
print("  own coordinates agree with city, PIN does not ->", int((own_ok & ~pin_ok).sum()))
print("  PIN agrees with city, own coordinates do not ->", int((~own_ok & pin_ok).sum()))
print("  neither agrees with city                      ->", int((~own_ok & ~pin_ok).sum()))
print("\nlocation source used for distance search (whole table):")
print(df.loc[~df["flag_empty"], "loc_source"].value_counts().to_string())
display(far[["name", "city", "pincode", "pin_district", "dist_to_pin_km", "own_to_city_km", "pin_to_city_km", "loc_source"]]
        .sort_values("dist_to_pin_km", ascending=False).head(15))

records where coordinates and PIN disagree: 1581
  own coordinates agree with city, PIN does not -> 690
  PIN agrees with city, own coordinates do not -> 526
  neither agrees with city                      -> 365

location source used for distance search (whole table):
loc_source
coordinates_match_pin     7947
coordinates_match_city     689
pin_matches_city           526
coordinates_unchecked      425
uncertain                  366
pin_only                    35
unknown                      9


name,city,pincode,pin_district,dist_to_pin_km,own_to_city_km,pin_to_city_km,loc_source
Ibn Sina Hospital,Budgam,191111,Budgam,10623.4,0.0,10623.4,coordinates_match_city
Krishna Netralaya,Gurugram,122018,Gurugram,8748.8,5.4,8743.5,coordinates_match_city
Avya Skin Clinic,Gurugram,122018,Gurugram,8746.2,5.7,8743.5,coordinates_match_city
Cloudnine Golf Course Road - Gurgaon,Gurugram,122018,Gurugram,8745.3,1.8,8743.5,coordinates_match_city
Dentaire Smile Studio,Gurugram,122018,Gurugram,8744.5,2.0,8743.5,coordinates_match_city
"Esthetica International Clinique - 95P First Floor, Sector 47, Gurgaon, Haryana - 122018, India, South City ii, Gurugram",Gurugram,122018,Gurugram,8744.2,0.6,8743.5,coordinates_match_city
Ministry of smile,Gurugram,122018,Gurugram,8744.0,1.9,8743.5,coordinates_match_city
Polymedica Dental & Medical Centre,Gurgaon,122018,Gurugram,8743.9,4.1,8743.3,coordinates_match_city
Bloom Women & Kids Clinic,Gurgaon,122018,Gurugram,8743.8,4.4,8743.3,coordinates_match_city
Meraki Skin Clinic,Gurugram,122018,Gurugram,8743.8,0.9,8743.5,coordinates_match_city


In [0]:
# README in Catalog: write a description on every column of the master table
# (runs after Step 5, so the descriptions are re-applied every time the table is rebuilt)
OUTPUT = "workspace.default.silver_facilities"
COMMENTS = {
    "unique_id": "Facility ID from the source (unique after removing 11 exact duplicate rows)",
    "name": "Facility name as scraped",
    "facility_type": "hospital / clinic / dentist / doctor / pharmacy; invalid raw values set to null",
    "operator_type": "private / public (government merged into public)",
    "description": "Free-text description; key evidence source, quote it in recommendations",
    "specialties": "De-duplicated list of claimed specialties (e.g. nephrology)",
    "procedure": "De-duplicated list of claimed procedures; often the most specific evidence",
    "equipment": "De-duplicated list of claimed equipment (e.g. dialysis machine, MRI scanner)",
    "capability": "De-duplicated list of capability statements (e.g. ICU available)",
    "n_specialties": "Number of distinct specialties",
    "n_doctors": "Number of doctors (numeric; mostly null)",
    "n_beds": "Number of beds (numeric; mostly null)",
    "year_established": "Founding year (1800-2026 kept)",
    "address": "Street address line 1",
    "city": "City as written, title case",
    "state": "CORRECTED state; replaced by the PIN state when the raw value was a city, district or alias",
    "state_raw": "Original state value before correction",
    "pincode": "6-digit PIN code extracted from the address",
    "pin_district": "Official district for the PIN (India Post directory, most common)",
    "pin_state": "Official state for the PIN (India Post directory)",
    "pin_n_districts": "Number of districts the PIN spans; above 1 means less precise",
    "lat": "Facility own latitude (null if outside India)",
    "lon": "Facility own longitude (null if outside India)",
    "pin_lat": "Latitude of the PIN area centre (median of its post offices)",
    "pin_lon": "Longitude of the PIN area centre",
    "dist_to_pin_km": "Distance in km between own coordinates and PIN centre",
    "loc_lat": "BEST latitude to use for distance search",
    "loc_lon": "BEST longitude to use for distance search",
    "loc_source": "Where loc_lat/loc_lon came from: coordinates_match_pin, coordinates_match_city, pin_matches_city, coordinates_unchecked, pin_only, uncertain, unknown",
    "official_phone": "Official phone number",
    "official_website": "Official website",
    "email": "Email address",
    "websites": "All websites linked to the record",
    "n_websites": "Number of websites; very long lists suggest merged, noisier evidence",
    "source_urls": "Web pages the record was extracted from",
    "flag_bad_coords": "True if own coordinates were outside India",
    "flag_state_mismatch": "True if the record names one valid state but its PIN is in another; possible merged record",
    "flag_coords_far_from_pin": "True if own coordinates and PIN centre are more than 50 km apart",
    "flag_location_uncertain": "True if the location conflict could not be settled; show location approximate",
    "flag_duplicate": "True if the same name appears more than once in the same city",
    "flag_mixed_identity": "True if text mixes a small single-system practice with large-hospital claims; likely merged",
    "flag_too_many_specialties": "True if 25+ specialties but under 50 beds; inflated list",
    "flag_implausible_counts": "True if doctor or bed numbers are unrealistic",
    "n_flags": "Total number of quality flags (0 = no issues found)",
    "source_table": "Bronze facilities table this row came from",
    "source_version": "Delta version of the Bronze facilities table used",
    "pin_source_table": "Bronze PIN directory table used for the district join",
    "pin_source_version": "Delta version of the PIN directory used",
    "silver_built_at": "UTC time this table was built",
    "silver_notebook": "Notebook that builds this table",
}
cols = set(spark.table(OUTPUT).columns)
missing = cols - set(COMMENTS)
assert not missing, f"columns without a description: {missing}"
for col, text in COMMENTS.items():
    spark.sql(f"ALTER TABLE {OUTPUT} ALTER COLUMN {col} COMMENT '{text}'")
print(f"described {len(COMMENTS)} of {len(cols)} columns in {OUTPUT}")

described 50 of 50 columns in workspace.default.silver_facilities
